# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BilaalBakare/Flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


The queue: what to do first, and why

Pages are ranked by impressions — the more visibility a page already has, the more potential upside if the underlying issue is fixed. Two reason codes cover distinct situations that need different treatment, not one blanket flag:

ZERO_CLICKS_WITH_VISIBILITY → review_content_quality — the page gets real impressions but literally zero clicks. This is treated separately from a CTR problem because zero clicks despite visibility can indicate a deeper issue (irrelevant content, wrong intent match, thin page) that a title/snippet tweak won't fix. This needs a human content review before any specific action is chosen.
POOR_POSITION_WITH_VISIBILITY → improve_ctr — position 21+, with CTR below the March portfolio average. This is the more mechanical case: the page ranks poorly but has clicks, so a title/snippet refinement is a reasonable first attempt.

Pages with healthy CTR relative to their position (like high-impression pages already converting well — flagged as weak picks in Week 4) are deliberately excluded from this queue, even if they have high impressions, because flagging already-healthy pages wastes review time and erodes trust in the queue.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from dotenv import load_dotenv
import os
import duckdb

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"


In [3]:
playbook_df = con.sql(f"""
    WITH base AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_avg_position,
            gsc_impressions,
            gsc_clicks,
            gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE month = '2026-03'
          AND gsc_avg_position IS NOT NULL
          AND gsc_impressions >= 50
    ),
    bucket_avg AS (
        SELECT AVG(ctr) AS overall_avg_ctr FROM base
    ),
    scored AS (
        SELECT
            base.*,
            bucket_avg.overall_avg_ctr,
            CASE
                WHEN base.gsc_clicks = 0 THEN 'ZERO_CLICKS_WITH_VISIBILITY'
                WHEN base.gsc_avg_position >= 21 AND base.ctr < bucket_avg.overall_avg_ctr THEN 'POOR_POSITION_WITH_VISIBILITY'
                ELSE NULL
            END AS reason_code,
            CASE
                WHEN base.gsc_clicks = 0 THEN 'review_content_quality'
                WHEN base.gsc_avg_position >= 21 AND base.ctr < bucket_avg.overall_avg_ctr THEN 'improve_ctr'
                ELSE NULL
            END AS action,
            base.gsc_impressions AS score
        FROM base, bucket_avg
    )
    SELECT *
    FROM scored
    WHERE reason_code IS NOT NULL
    ORDER BY score DESC
""").df()

playbook_df.head(20)

,client_hash_id,content_hash_id,report_date,gsc_avg_position,gsc_impressions,gsc_clicks,ctr,overall_avg_ctr,reason_code,action,score
0,client_62f4a7e64f5e0096,content_945d6ff91386c817,2026-03-04,8.613948,37368,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,37368
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-30,0.181500,33383,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,33383
2,client_23a62021009f63c4,content_44f34c0a90047651,2026-03-27,0.132532,32958,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,32958
3,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-31,0.083407,31472,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,31472
4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-02,0.000311,28973,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,28973
5,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-01,0.002245,28947,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,28947
6,client_62f4a7e64f5e0096,content_34a70fea29d15f24,2026-03-22,3.129405,27410,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,27410
7,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,2026-03-03,0.317996,24233,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,24233
8,client_e547b89c05043229,content_757b1fa67827358d,2026-03-13,2.261644,19301,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,19301
9,client_73cda7b4e4f265ea,content_fec55986a1868d62,2026-03-24,0.013056,17770,0,0.0,0.00306,ZERO_CLICKS_WITH_VISIBILITY,review_content_quality,17770


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Who uses this: A content strategist or SEO lead, as a starting point for prioritizing manual review — not an automated action list. It answers "where should I look first?", not "what should I do without checking?"

What it's for: Surfacing pages that already have real search visibility (meaningful impressions) but aren't converting that visibility into clicks, so limited review time goes to the pages with the most recoverable upside first.

Where it stops being valid:

Single client, single month. This slice is drawn almost entirely from one client (client_23a62021009f63c4, confirmed in Week 4) over March 2026 only. It has not been validated against other clients, other months, or seasonal variation — a different client or time period could show a different pattern entirely.
No causal claim. The queue reflects an observed, directional association between position/CTR and visibility — not proof that applying the recommended action will fix the page. Whether a fix actually works would need a real before/after measurement, which this project has not run.
Partial feature/target overlap (Week 6 finding). gsc_impressions is used both as a ranking feature and as part of how the underlying CTR benchmark is computed — a structural overlap that means scores should be read as directional priority, not a precise, independent probability.
No revenue or conversion data. The queue optimizes for clicks recovered, not business value — a low-value page with high impressions could rank above a high-value page with fewer impressions.
Data currency. GA4, session-source, and AI-referral columns were entirely empty in this slice (Week 3 finding) — this queue reflects Search Console signals only, and says nothing about actual on-site engagement or conversion behavior.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.